In [9]:
!pip install -U peft bitsandbytes transformers accelerate

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [11]:
import os
print(os.getcwd())

/content


In [12]:
import os

print(os.listdir("/content/drive/MyDrive"))

['Lecture7.pptx', 'Colab Notebooks', "Amir's Resume.pdf", 'github-recovery-codes.txt', 'Untitled document (1).gdoc', 'Untitled document.gdoc', 'loxmonlufndlyh0lmx3.svg', 'Studets Grade-1.gsheet', 'To-do list.gsheet', 'Untitled spreadsheet (1).gsheet', 'Untitled spreadsheet.gsheet', 'AI PhD Thesis Topic Selection.gdoc', 'LinkedIn Profile AI Agent Pivot.gsheet', 'Metformin.pdf']


In [13]:
file_path = "/content/drive/MyDrive/Metformin.pdf"
with open(file_path, "rb") as f:
    print("File loaded successfully ✅")

File loaded successfully ✅


In [14]:
pip install -U peft bitsandbytes transformers accelerate trl PyMuPDF

In [15]:
!pip install -U trl

In [16]:
!pip install PyMuPDF

In [17]:
from datasets import Dataset, load_dataset

In [18]:
dataset = load_dataset("roneneldan/TinyStories", split="train")

In [19]:
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 2119719
})


In [20]:
print(dataset[10])

{'text': 'Once upon a time, there was a big car named Dependable. He had a very important job. Dependable would take a family to the park every day. The family had a mom, dad, and a little girl named Lily. They all had a lot of love for each other.\n\nOne day, when they got to the park, they saw a big sign that said, "Fun Race Today!" The family was very excited. They knew that Dependable was very fast and could win the race. So, they decided to join the race.\n\nThe race started, and Dependable went very fast. The other cars tried to catch up, but Dependable was too quick. In the end, Dependable won the race! The family was so happy and proud of their car. They knew that their love for each other and their trust in Dependable made them win the race. And from that day on, they had even more fun at the park, knowing that they had the fastest and most dependable car around.'}


In [21]:
import fitz
import os


In [22]:


def extract_text_from_pdf(pdf_path):
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"The file '{pdf_path}' does not exist.")
    
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text()
            text_blocks.append(text)
    return text_blocks

In [23]:
import os
print("Current Folder:", os.getcwd())
print("Files in Colab:", os.listdir('.'))


Current Folder: /content
Files in Colab: ['.config', 'llama-pharma-domain', 'drive', 'sample_data']


In [24]:
# 1. Click the 'Files' icon in the left sidebar of Colab/VS Code
# 2. Drag and drop 'LLM-Finetuning/LLM Fine-Tuning-Train-LLMs-on-Your-PDF-Text-Data -Domain-Specific-Fine-Tuning-with-HuggingFace/Metformin.pdf' into it
# 3. Then run this cell
pdf_texts = extract_text_from_pdf('/content/drive/MyDrive/Metformin.pdf')
print(f'Extracted {len(pdf_texts)} pages.')

Extracted 1 pages.


In [25]:
pdf_texts

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis. \n \nClinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA red

In [26]:
import re
def split_paragraphs(pages):
    paragraphs = []
    for page_text in pages:
        # Split on double line breaks or long newlines
        chunks = re.split(r'\n\s*\n', page_text)
        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30:  # ignore too short lines
                paragraphs.append(clean)
    return paragraphs

In [27]:
paragraphs = split_paragraphs(pdf_texts)

In [28]:
data = [{"text": p} for p in paragraphs]
data

[{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.'},
 {'text': 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits h

In [29]:
print(data)

[{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.'}, {'text': 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits he

In [30]:
dataset = Dataset.from_list(data)

In [23]:
dataset

Dataset({
    features: ['text'],
    num_rows: 4
})

In [24]:
# model_name = "meta-llama/Llama-2-7b-hf"
# model_name = "meta-llama/Llama-3.1-8B"
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [26]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [27]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [28]:
def tokenize_fn(examples):
    tokens = tokenizer(examples["text"],truncation=True,padding="max_length",max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [29]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

tokenized

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4
})

In [30]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [31]:
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [32]:
training_args = TrainingArguments(
    output_dir="./llama-pharma-domain",
    # overwrite_output_dir=True,  <-- REMOVE OR COMMENT OUT THIS LINE
    num_train_epochs=2,
    per_device_train_batch_size=2,
    # ... your other arguments
)


In [33]:
from transformers import TrainingArguments
help(TrainingArguments)

Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing

In [36]:
training_args = TrainingArguments(
    output_dir="./llama-pharma-domain",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)


In [38]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized
)

In [39]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


: 

: 

: 

### Now lets see the LORA based method

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
!pip install -U peft bitsandbytes transformers accelerate

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [1]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [5]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [31]:
def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

In [32]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [34]:
from transformers import BitsAndBytesConfig


In [36]:
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model,  # Ensure this contains the model string path
    quantization_config=quantization_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [37]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [39]:
# save adapter
non_inst_model_lora = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [40]:
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [41]:
trainer = Trainer(
    model=non_inst_model_lora,
    args=args,
    train_dataset=tokenized
)

In [42]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


TrainOutput(global_step=5, training_loss=9.342776489257812, metrics={'train_runtime': 907.5225, 'train_samples_per_second': 0.022, 'train_steps_per_second': 0.006, 'total_flos': 63629646888960.0, 'train_loss': 9.342776489257812, 'epoch': 5.0})

In [43]:
model_path = "/content/tinyllama-lora/checkpoint-5"

In [44]:
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [45]:
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

In [47]:
inputs = tokenizer(prompt, return_tensors="pt").to("cpu")


In [48]:
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [49]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe reduced LDL-C and triglycerides by 25% to 37%, compared with a statin alone.13
Another drug approved in 2012 for the treatment of AMI, the novel lipid-lowering agent, Prilosec ODT (omeprazole/dapoxetine hydrochloride), also received FDA approval for the management of HF patients in 2012.14
